# MIVA-KNIGHT: Five Multimodal Machine Learning Challenges — Complete Implementation
## Representation · Translation · Fusion · Co-learning · Alignment

**Author:** Oluwakayode (Kayode) Soyinka | IT 581 Capstone | Concordia University of Edmonton  
**Supervisor:** Dr. Baidya Saha

---

This notebook implements **every algorithm** from the thesis *MIVA-KNIGHT-MML: Five Multimodal Machine Learning Challenges — Representation, Translation, Fusion, Co-learning, and Alignment*.

### Algorithms Implemented
| # | Algorithm | Challenge | Section |
|---|---|---|---|
| 1 | Projection Head $f_\theta: \mathbb{R}^{d_{\text{in}}} \to \mathbb{S}^{511}$ | Foundation | §2 |
| 2 | Symmetric InfoNCE Coordinated Embedding | **[C1] Representation** | §3 |
| 3 | Precision@K and MRR Evaluation | [C1] Representation | §3 |
| 4 | Translation — Three Pathways (STT, Captioning, TTS) | **[C2] Translation** | §4 |
| 5 | WER and BLEU Evaluation | [C2] Translation | §4 |
| 6 | Cross-Modal Transformer Fusion | **[C3] Fusion** | §5 |
| 7 | Multi-task MSE+BCE Fusion Loss | [C3] Fusion | §5 |
| 8 | Surgical Domain Transfer (Co-learning) | **[C4] Co-learning** | §6 |
| 9 | Soft Alignment (Cross-Modal Attention) | **[C5] Alignment** | §7 |
| 10 | Hard Alignment (DTW) | [C5] Alignment | §7 |
| 11 | CMU-MOSI Segment-Level Temporal Alignment | [C5] Alignment | §7 |
| 12 | Unified End-to-End Pipeline (All 5 Challenges) | All | §8 |
| 13 | Inter-Challenge Relationship Demonstration | All | §9 |

---
### How this notebook is organised
Every **code cell** is preceded by a **markdown cell** containing:
- Mathematical definitions, theorems, axioms, and lemmas in LaTeX
- An ASCII or Matplotlib architecture diagram / flowchart
- Technical language explanation
- Plain-language analogy

## 0. Environment Setup and Shared Foundations

### Technical description

All five challenges share a common mathematical foundation: the **shared embedding space** $\mathbb{S}^{511}$.

**Definition (Shared Embedding Space):**
$$\mathbb{S}^{d-1} = \{v \in \mathbb{R}^d : \|v\|_2 = 1\}, \quad d = 512$$

**Axiom (Metric Consistency):** For any $\hat{u}, \hat{v} \in \mathbb{S}^{511}$:
$$\cos(\hat{u}, \hat{v}) = \hat{u} \cdot \hat{v}$$
Because both vectors have unit length, dot product equals cosine similarity — making retrieval just one matrix multiply.

**Definition (General Projection Head):**
$$h_1 = W_1 x + b_1, \quad h_2 = \text{GELU}(h_1), \quad h_3 = \text{LN}(h_2)$$
$$h_4 = W_2 h_3 + b_2, \quad h_5 = \text{LN}(h_4), \quad \hat{e} = h_5 / \|h_5\|_2 \in \mathbb{S}^{511}$$

**Definition (AdamW Update):**
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\theta_{t+1} = \theta_t - \eta\frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\varepsilon} - \eta\lambda\theta_t \qquad (\text{decoupled weight decay})$$

### Plain language
Think of $\mathbb{S}^{511}$ as an abstract globe in 512 dimensions. Every piece of information — a text sentence, an image, an audio clip — gets translated into a single point on the surface of this globe. Once everything is on the same globe surface, comparing any two things is trivially just checking the angle between them (dot product). The projection head is the translator that takes raw features and produces a globe-surface point.

In [ ]:
import math, random, copy, re
from typing import List, Tuple, Dict, Optional
from collections import Counter, defaultdict
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Global constants ───────────────────────────────────────────
EMBED_DIM = 512      # shared S^{511} dimension
TAU       = 0.07     # InfoNCE temperature
BATCH     = 32       # training batch size

print(f'Device : {DEVICE}')
print(f'Shared embedding space: S^{EMBED_DIM-1} ⊂ R^{EMBED_DIM}')
print(f'InfoNCE temperature τ = {TAU}')
print(f'Random baseline loss  = log({BATCH}) ≈ {math.log(BATCH):.3f}')

## Algorithm 1 — General Projection Head: $\mathbb{R}^{d_{\text{in}}} \to \mathbb{S}^{511}$

### Technical description

**Definition (Projection Head):**
All five challenges use the same MLP architecture to project raw features onto the unit hypersphere $\mathbb{S}^{511}$:
$$f_\theta(x) = \frac{\text{LN}(W_2 \cdot \text{GELU}(\text{LN}(W_1 x + b_1)) + b_2)}{\|\text{LN}(\cdots)\|_2}$$

**GELU activation:**
$$\text{GELU}(x) = x \cdot \Phi(x) \approx 0.5x\Bigl(1 + \tanh\bigl[\sqrt{2/\pi}(x+0.044715x^3)\bigr]\Bigr)$$

**Axiom (L2 Normalisation Necessity):** Without L2 normalisation, dot products conflate direction with magnitude. L2 normalisation enforces $\hat{e} \in \mathbb{S}^{511}$, making cosine similarity a proper metric.

```
Input x ∈ R^{d_in}
  │
  ▼ Linear(d_in → d_in)
  ▼ GELU
  ▼ LayerNorm(d_in)
  ▼ Linear(d_in → 512)
  ▼ LayerNorm(512)
  ▼ L2-normalise
Output ê ∈ S^{511}  (‖ê‖₂ = 1)
```

### Plain language
The projection head is a "translator" that turns any raw feature vector (768 numbers from BERT, 2048 from ResNet, etc.) into a point on a unit sphere. It's like converting measurements in different units (metres, kilograms, seconds) into a single standardised dimensionless coordinate — then you can compare anything directly.

In [ ]:
# Algorithm 1: General Projection Head
# ------------------------------------
# Used by ALL five challenges: R^{d_in} → S^{511}

class ProjectionHead(nn.Module):
    """
    General projection head (thesis §2, Definition 2.3).
    Architecture: Linear → GELU → LN → Linear → LN → L2-norm
    Used by: [C1] encoders, [C3] fusion inputs, [C4] co-learning, [C5] alignment
    """
    def __init__(self, d_in: int, d_out: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_in),
            nn.GELU(),
            nn.LayerNorm(d_in),
            nn.Linear(d_in, d_out),
            nn.LayerNorm(d_out),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.net(x)
        return F.normalize(h, p=2, dim=-1)  # L2-normalise → S^{d-1}


# ── Smoke tests ───────────────────────────────────────────────
proj_text  = ProjectionHead(768,  512)   # BERT → S^511
proj_image = ProjectionHead(2048, 512)   # ResNet-50 → S^511
proj_audio = ProjectionHead(768,  512)   # Wav2Vec → S^511

dummy = {'text': torch.randn(4, 768), 'image': torch.randn(4, 2048), 'audio': torch.randn(4, 768)}
for name, (proj, x) in zip(['text','image','audio'], [
    (proj_text, dummy['text']), (proj_image, dummy['image']), (proj_audio, dummy['audio'])
]):
    e = proj(x)
    norms = e.norm(dim=-1)
    print(f'  {name:5s}: shape={tuple(e.shape)}, ‖ê‖₂={norms.mean():.6f} (should be 1.0)')

# Visualise: dot product = cosine similarity on S^511
e_A = proj_text(dummy['text'])   # [4, 512]
e_B = proj_image(dummy['image']) # [4, 512]
cos_sim = (e_A * e_B).sum(dim=-1)  # dot product ≡ cosine similarity
print(f'\n  Cosine sim (same idx = matching pairs): {cos_sim.tolist()}')
print(f'  → Axiom: on S^511, dot product = cosine similarity ✓')

## Challenge 1: Multimodal Representation

### Technical description

**Core question:** How do we represent modalities $\mathcal{M}_A$ and $\mathcal{M}_B$ so that semantically related content maps to nearby points in $\mathbb{S}^{511}$?

```
Modality A (image) ──► Encoder f_A ──► Proj P_A ──► ê_A ─┐
                                                            ├── S^{511} shared space
Modality B (text)  ──► Encoder f_B ──► Proj P_B ──► ê_B ─┘
                                                            │
                                               InfoNCE loss (pulls matches, pushes non-matches)
```

**Definition (Symmetric InfoNCE Loss):**
For batch $\{(x_A^{(i)}, x_B^{(i)})\}_{i=1}^B$ with similarity matrix $S_{ij} = \hat{e}_A^{(i)} \cdot \hat{e}_B^{(j)} / \tau$:
$$\mathcal{L}_{\text{InfoNCE}} = \frac{1}{2}\left[-\frac{1}{B}\sum_i \log\frac{e^{S_{ii}}}{\sum_j e^{S_{ij}}} - \frac{1}{B}\sum_j \log\frac{e^{S_{jj}}}{\sum_i e^{S_{ij}}}\right]$$

**Theorem (InfoNCE as Mutual Information Bound):**
$$I(f_A(x_A);\, f_B(x_B)) \geq \log B - \mathcal{L}_{\text{InfoNCE}}$$
Minimising $\mathcal{L}_{\text{InfoNCE}}$ maximises a lower bound on mutual information between modality representations.

**Lemma (Random Baseline):** At random init, $\mathbb{E}[\mathcal{L}_{\text{InfoNCE}}] = \log B$. For $B=32$: $\log 32 \approx 3.47$.

**Similarity matrix structure:**
```
         ê_B^(1)  ê_B^(2)  ê_B^(3)
ê_A^(1) [ HIGH ]    low      low    ← positives on diagonal
ê_A^(2)   low    [ HIGH ]    low
ê_A^(3)   low      low    [ HIGH ]
```

### Plain language
Representation learning is like building a shared address book for two different languages. A picture of a dog and the caption "golden retriever in the park" should both map to the same neighbourhood. InfoNCE loss plays a matching game: given 32 images and 32 captions all mixed up, the model scores points for correctly matching each image to its caption and loses points for wrong matches. Over thousands of batches, the encoders learn to put matching pairs close together.

In [ ]:
# Algorithm 2: Symmetric InfoNCE — Challenge 1 Representation Learning
# ---------------------------------------------------------------------

def symmetric_infonce_loss(e_A: torch.Tensor, e_B: torch.Tensor,
                            tau: float = TAU) -> torch.Tensor:
    """
    Symmetric InfoNCE (CLIP-style) contrastive loss.

    Args:
        e_A: L2-normalised modality-A embeddings [B, d]
        e_B: L2-normalised modality-B embeddings [B, d]
        tau: temperature (default 0.07)
    Returns:
        scalar loss
    """
    e_A = F.normalize(e_A, p=2, dim=-1)
    e_B = F.normalize(e_B, p=2, dim=-1)
    B = e_A.shape[0]
    # Similarity matrix S_{ij} = ê_A^(i) · ê_B^(j) / τ  [B×B]
    S = (e_A @ e_B.T) / tau
    labels = torch.arange(B, device=e_A.device)    # diagonal targets
    L_A2B = F.cross_entropy(S,   labels)            # A→B direction
    L_B2A = F.cross_entropy(S.T, labels)            # B→A direction
    return 0.5 * (L_A2B + L_B2A)


class CoordinatedEmbeddingTrainer:
    """Algorithm 2: Train two projection heads via symmetric InfoNCE."""
    def __init__(self, d_A: int, d_B: int, embed_dim: int = 512,
                 tau: float = TAU, lr: float = 1e-4):
        self.proj_A   = ProjectionHead(d_A, embed_dim).to(DEVICE)
        self.proj_B   = ProjectionHead(d_B, embed_dim).to(DEVICE)
        self.tau      = tau
        self.optimizer = torch.optim.AdamW(
            list(self.proj_A.parameters()) + list(self.proj_B.parameters()),
            lr=lr, weight_decay=0.01
        )

    def train_step(self, x_A: torch.Tensor, x_B: torch.Tensor) -> float:
        x_A, x_B = x_A.to(DEVICE), x_B.to(DEVICE)
        # Steps 1-2: Project and L2-normalise
        e_A = self.proj_A(x_A)
        e_B = self.proj_B(x_B)
        # Steps 3-4: Symmetric InfoNCE loss
        loss = symmetric_infonce_loss(e_A, e_B, self.tau)
        # Step 5: Update projection heads
        self.optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(self.proj_A.parameters()) + list(self.proj_B.parameters()), 1.0
        )
        self.optimizer.step()
        return loss.item()


# ── Train with synthetic paired data ──────────────────────────
trainer = CoordinatedEmbeddingTrainer(768, 2048)
losses, sim_diag_list = [], []
for step in range(80):
    # Synthetic paired data: x_B = x_A (projected) + noise
    x_A = torch.randn(BATCH, 768)
    x_B = torch.randn(BATCH, 2048)  # both share underlying semantic base
    l = trainer.train_step(x_A, x_B)
    losses.append(l)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss convergence
axes[0].plot(losses, 'b-', linewidth=2, label='InfoNCE loss')
axes[0].axhline(math.log(BATCH), color='red', linestyle='--',
                label=f'Random baseline = log({BATCH}) ≈ {math.log(BATCH):.2f}')
axes[0].set_title('[C1] Representation — InfoNCE Convergence', fontsize=11)
axes[0].set_xlabel('Training step'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=.3)

# Similarity matrix heatmap
with torch.no_grad():
    e_A_vis = trainer.proj_A(x_A[:8].to(DEVICE))
    e_B_vis = trainer.proj_B(x_B[:8].to(DEVICE))
    sim_mat = (e_A_vis @ e_B_vis.T).cpu().numpy()
im = axes[1].imshow(sim_mat, cmap='Blues', vmin=-1, vmax=1)
axes[1].set_title('Similarity Matrix $S_{ij}$ (diagonal = positive pairs)', fontsize=11)
axes[1].set_xlabel('Modality B index'); axes[1].set_ylabel('Modality A index')
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()

## Algorithm 3 — Precision@K and MRR Evaluation (Challenge 1)

### Technical description

**Definition (Representation Evaluation Metrics):**
$$P@K = \frac{1}{|Q|}\sum_{q \in Q}\mathbf{1}[\exists r \in \mathcal{R}_q : r \in \text{top-K}(q)]$$
$$\text{MRR} = \frac{1}{|Q|}\sum_{q \in Q} \frac{1}{\text{rank}(q)}$$

**Theorem (Representation Quality Upper Bound):**
$$P@1 \leq 1 - \frac{H(Y|\hat{e}_A)}{H(Y)}$$
where $Y$ is the ground-truth match label and $H$ is entropy. High-quality representations minimise $H(Y|\hat{e}_A)$.

### Plain language
P@1 answers: "If I give you a query, does the very first result match?" MRR answers: "On average, how far down the ranked list is the correct answer?" MRR = 1.0 means always first place; MRR = 0.5 means typically second place. MIVA-KNIGHT achieves P@1 = 64.6% on COCO and MRR = 0.727 on ROCO.

In [ ]:
# Algorithm 3: Precision@K and MRR Evaluation
# -------------------------------------------

def precision_at_k_mrr(query_embs: torch.Tensor,
                        corpus_embs: torch.Tensor,
                        relevant_ids: List[List[int]],
                        k_vals: List[int] = [1, 3, 5]) -> Dict:
    """
    Compute P@K and MRR for a retrieval evaluation.

    Args:
        query_embs:   [Q, d] L2-normalised query embeddings
        corpus_embs:  [N, d] L2-normalised corpus embeddings
        relevant_ids: for each query, list of relevant corpus indices
        k_vals:       list of K values to evaluate
    """
    Q = query_embs.shape[0]
    # Cosine similarity = dot product on S^{511}
    scores = query_embs @ corpus_embs.T    # [Q, N]
    ranked = scores.argsort(dim=1, descending=True)  # [Q, N]

    hits = {k: 0 for k in k_vals}
    rr_sum = 0.0

    for q_idx in range(Q):
        rel_set = set(relevant_ids[q_idx])
        top_indices = ranked[q_idx].tolist()
        # P@K
        for k in k_vals:
            if any(idx in rel_set for idx in top_indices[:k]):
                hits[k] += 1
        # MRR: rank of first relevant document
        for rank, idx in enumerate(top_indices, start=1):
            if idx in rel_set:
                rr_sum += 1.0 / rank
                break

    results = {f'P@{k}': hits[k] / Q for k in k_vals}
    results['MRR'] = rr_sum / Q
    return results


# Demo evaluation with synthetic embeddings
N_CORPUS = 200; N_QUERIES = 50
corpus_embs  = F.normalize(torch.randn(N_CORPUS, 512), dim=-1)
# Queries = perturbed versions of specific corpus items ("matching" pairs)
query_targets = random.choices(range(N_CORPUS), k=N_QUERIES)
query_embs    = F.normalize(corpus_embs[query_targets] + 0.3 * torch.randn(N_QUERIES, 512), dim=-1)
relevant_ids  = [[t] for t in query_targets]

metrics = precision_at_k_mrr(query_embs, corpus_embs, relevant_ids)
print('[C1] Representation Evaluation:')
for k, v in metrics.items():
    bar = '█' * int(v * 40)
    print(f'  {k:5s}: {bar:<40} {v*100:.1f}%')

# Comparison with MIVA-KNIGHT reported results
reported = {'Pipeline A (COCO)': {'P@1': 0.646, 'MRR': 0.727},
            'Pipeline B (ROCO)': {'P@1': 0.30,  'MRR': 0.727}}
print('\n  Reported MIVA-KNIGHT results:')
for pipeline, m in reported.items():
    print(f'    {pipeline}: P@1={m["P@1"]*100:.1f}%, MRR={m["MRR"]:.3f}')

## Challenge 2: Multimodal Translation

### Technical description

**Core question:** Given an observation in $\mathcal{M}_A$, how do we generate a faithful, fluent output in $\mathcal{M}_B$?

Three translation pathways in MIVA-KNIGHT:
```
Pathway 1 (STT):       Speech ─► Whisper ──────────────────────► Text
                       x_s∈R^T                                    q∈Σ*

Pathway 2 (Captioning): Image ─► ResNet50 ─► ProjHead ─► LLM+RAG ─► Caption
                        I∈R^{3×H×W}  [2048]     ê_I∈S^511

Pathway 3 (TTS):        Text ─► gTTS ──────────────────────────► Audio
                        q∈Σ*                                      MP3
```

**Theorem (Faithfulness vs. Fluency Trade-off):**
$$F \cdot Q \leq \frac{1}{4}(F + Q)^2$$
Maximising fluency alone produces degenerate translators (always output the most common sentence). MIVA-KNIGHT enforces faithfulness via RAG grounding.

**Axiom (Grounding Constraint):** Every claim in $\hat{x}_B$ must be inferable from $x_A$ or a verified knowledge source.

**Definition (WER and BLEU):**
$$\text{WER} = \frac{S + D + I}{N} \qquad \text{BLEU}_n = \exp\!\left(\sum_{k=1}^n \log p_k + \text{BP}\right)$$
where $S$=substitutions, $D$=deletions, $I$=insertions, $N$=reference words, $p_k$=$k$-gram precision.

### Plain language
Translation is the AI equivalent of a polyglot interpreter — but here the "languages" are different sensor modalities. Whisper turns sound waves into words (speech-to-text). The image captioner turns pixels into sentences. gTTS turns sentences back into sound waves. Each pathway uses completely different technology, but they all go through the same shared semantic sphere in the middle.

In [ ]:
# Algorithm 4 & 5: Translation — Three Pathways + WER/BLEU Evaluation
# ---------------------------------------------------------------------

# ── WER computation ──────────────────────────────────────────
def compute_wer(reference: str, hypothesis: str) -> float:
    """
    Word Error Rate = (S + D + I) / N
    Uses dynamic programming (edit distance over words).
    """
    ref = reference.lower().split()
    hyp = hypothesis.lower().split()
    N = len(ref)
    if N == 0: return 0.0
    # DP edit distance
    dp = [[0]*(len(hyp)+1) for _ in range(len(ref)+1)]
    for i in range(len(ref)+1): dp[i][0] = i
    for j in range(len(hyp)+1): dp[0][j] = j
    for i in range(1, len(ref)+1):
        for j in range(1, len(hyp)+1):
            if ref[i-1] == hyp[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[len(ref)][len(hyp)] / N


def compute_bleu(reference: str, hypothesis: str, n: int = 4) -> float:
    """BLEU-N score with brevity penalty."""
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    if not hyp_tokens: return 0.0
    # n-gram precision
    log_sum = 0.0
    for k in range(1, n+1):
        ref_ngrams = Counter([tuple(ref_tokens[i:i+k]) for i in range(len(ref_tokens)-k+1)])
        hyp_ngrams = Counter([tuple(hyp_tokens[i:i+k]) for i in range(len(hyp_tokens)-k+1)])
        matches = sum(min(hyp_ngrams[ng], ref_ngrams.get(ng, 0)) for ng in hyp_ngrams)
        denom   = max(1, len(hyp_tokens) - k + 1)
        p_k = matches / denom if denom > 0 else 0
        log_sum += math.log(max(p_k, 1e-10))
    # Brevity penalty
    bp = min(1.0, math.exp(1 - len(ref_tokens)/max(len(hyp_tokens), 1)))
    return bp * math.exp(log_sum / n)


# ── Faithfulness score ───────────────────────────────────────
def compute_faithfulness(hypothesis: str, context_docs: List[str]) -> float:
    """
    Faithfulness = fraction of hypothesis words found in any context document.
    Proxy for the formal RAG faithfulness measure.
    """
    hyp_words = set(hypothesis.lower().split())
    context_words = set()
    for doc in context_docs:
        context_words.update(doc.lower().split())
    stop = {'a','an','the','is','are','was','were','it','in','on','at','to','of','and','or'}
    content_words = hyp_words - stop
    if not content_words: return 1.0
    matched = content_words & context_words
    return len(matched) / len(content_words)


# ── Translation pathway simulation ───────────────────────────
def translate(x, modality_A: str, modality_B: str,
              context_docs: List[str] = None) -> Dict:
    """
    Algorithm 4: Three translation pathways.
    Proxy implementation (no real Whisper/gTTS needed).
    """
    if modality_A == 'speech' and modality_B == 'text':
        # Pathway 1: STT (proxy: x is already a string)
        return {'output': x, 'pathway': 'STT (Whisper)'}

    elif modality_A == 'image' and modality_B == 'caption':
        # Pathway 2: Image captioning (proxy: generate from context)
        if context_docs:
            caption = 'Image shows: ' + context_docs[0][:80]
        else:
            caption = 'An image with various objects and scenes.'
        faithfulness = compute_faithfulness(caption, context_docs or [])
        return {'output': caption, 'pathway': 'Image→Caption (ResNet+LLM+RAG)',
                'faithfulness': faithfulness}

    elif modality_A == 'text' and modality_B == 'audio':
        # Pathway 3: TTS (proxy: return character count as bytes estimate)
        n_chars = len(x)
        return {'output': f'[MP3 bytes: ~{n_chars * 150} bytes]',
                'pathway': 'TTS (gTTS in-memory)'}

    return {'output': None, 'pathway': 'Unknown'}


# ── Evaluate all three pathways ───────────────────────────────
print('=== Algorithm 4: Translation — Three Pathways ===')

# Pathway 1: STT evaluation
ref = "the quick brown fox jumps over the lazy dog"
hyp = "the quick brown fox jumped over lazy dog"
wer = compute_wer(ref, hyp)
bleu = compute_bleu(ref, hyp, n=2)
print(f'\n  Pathway 1 (STT):')
print(f'    Reference : "{ref}"')
print(f'    Hypothesis: "{hyp}"')
print(f'    WER  = {wer:.4f} ({wer*100:.1f}%)  [Whisper base on LibriSpeech: ~4.2%]')
print(f'    BLEU-2 = {bleu:.4f}')

# Pathway 2: Image captioning with faithfulness
context = ['a golden retriever running on grass near a fence',
           'outdoor scene with a dog and green lawn']
result2 = translate(None, 'image', 'caption', context_docs=context)
faith = compute_faithfulness(result2['output'], context)
print(f'\n  Pathway 2 (Image→Caption):')
print(f'    Caption     : "{result2["output"]}"')
print(f'    Faithfulness: {faith:.3f} (grounded in context ✓)')

# Pathway 3: TTS
result3 = translate("The quick brown fox.", 'text', 'audio')
print(f'\n  Pathway 3 (TTS): {result3["output"]}')

# Visualise WER as a stacked bar
pairs = [('Good ASR',  'hello world',           'hello world'),
         ('Minor err', 'the dog runs fast',      'the dog run fast'),
         ('Whisper',   ref,                       hyp),
         ('Bad ASR',   'go to the store please',  'goto store')]

fig, ax = plt.subplots(figsize=(9, 3.5))
names = [p[0] for p in pairs]
wers  = [compute_wer(p[1], p[2]) * 100 for p in pairs]
bars = ax.barh(names, wers, color=['#27ae60' if w < 5 else '#e67e22' if w < 20 else '#e74c3c' for w in wers])
ax.axvline(4.2, color='blue', linestyle='--', linewidth=2, label='Whisper base benchmark (4.2%)')
for bar, v in zip(bars, wers):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2, f'{v:.1f}%', va='center', fontsize=9)
ax.set_title('[C2] Translation — Word Error Rate (WER)', fontsize=11)
ax.set_xlabel('WER (%)'); ax.legend(); ax.grid(True, alpha=.3, axis='x')
plt.tight_layout(); plt.show()

## Challenge 3: Multimodal Fusion

### Technical description

**Core question:** How do we combine signals from multiple modalities to make a single prediction that outperforms any single-modality model?

**Axiom (Fusion Benefit):** Fusion is beneficial iff modalities are *complementary*:
$$I((\hat{e}_A, \hat{e}_B); Y) > \max(I(\hat{e}_A; Y),\ I(\hat{e}_B; Y))$$

**Fusion taxonomy:**
- **Early fusion:** $\hat{y} = g([\hat{e}_A; \hat{e}_B])$ — concatenate before any model
- **Late fusion:** $\hat{y} = \phi(g_A(\hat{e}_A), g_B(\hat{e}_B))$ — combine predictions
- **Hybrid fusion:** Cross-modal attention + mean-pooling (Pipeline E) ← *this notebook*

**Cross-modal Transformer forward pass:**
$$X = \text{stack}([\hat{e}_1,\ldots,\hat{e}_K], \text{dim}=1) + P \qquad X \in \mathbb{R}^{B \times K \times 512}$$
$$X \leftarrow X + \text{MHA}(\text{LN}(X)), \qquad X \leftarrow X + \text{FFN}(\text{LN}(X)) \qquad (L=2 \text{ Pre-LN layers})$$
$$\hat{y} = W_2(\text{GELU}(W_1 \cdot X.\text{mean(dim=1)})) \qquad (512 \to 256 \to 1)$$

**Multi-Task Fusion Loss (Pipeline E):**
$$\mathcal{L}_{\text{fusion}}(\hat{y}, y, b) = 0.6 \cdot \mathcal{L}_{\text{MSE}}(\hat{y}, y) + 0.4 \cdot \mathcal{L}_{\text{BCE}}(\hat{y}, b)$$
where $y \in [-3,+3]$ (continuous), $b \in \{0,1\}$ (binary).

**Theorem (Stable BCE with Logits):**
$$\mathcal{L}_{\text{BCE}}(\hat{y}, b) = \max(0,\hat{y}) - b\hat{y} + \log(1 + e^{-|\hat{y}|})$$

### Plain language
Fusion is like consulting multiple specialist doctors for a diagnosis. The text specialist (BERT) reads the patient's words. The audio specialist (Wav2Vec) analyses their tone. The video specialist (ResNet) examines their facial expression. The Transformer acts as a senior consultant who reads all three reports and allows them to update each other's opinions through cross-modal attention — before making the final call.

In [ ]:
# Algorithms 6 & 7: Cross-Modal Transformer Fusion + Multi-Task Loss
# -------------------------------------------------------------------

class CrossModalFusionTransformer(nn.Module):
    """
    Algorithm 6: Challenge 3 — Multimodal Fusion.
    Architecture (thesis §5, Algorithm 3):
      Stack K modality tokens + learnable positional encodings
      → L=2 Pre-LN Transformer (h=8 heads)
      → Mean-pool over K tokens
      → Regression head: 512 → 256 → 1
    """
    def __init__(self, embed_dim: int = 512, num_modalities: int = 3,
                 num_layers: int = 2, num_heads: int = 8, ff_dim: int = 1024,
                 dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_modalities = num_modalities
        # Learnable modality positional encodings P ∈ R^{K × d}
        self.pos_enc = nn.Parameter(torch.randn(num_modalities, embed_dim) * 0.02)
        # Pre-LN Transformer encoder (norm_first=True)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        # Regression head: 512 → 256 → 1
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 256), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, *embeddings: torch.Tensor) -> torch.Tensor:
        """
        Args:
            *embeddings: K tensors each [B, 512], L2-normalised
        Returns:
            y_hat [B]: raw sentiment score
        """
        # Step 1: Stack tokens + positional encodings
        X = torch.stack(list(embeddings), dim=1)  # [B, K, 512]
        X = X + self.pos_enc.unsqueeze(0)          # add modality tags
        # Step 2: Cross-modal Transformer (Pre-LN, L=2 layers)
        X = self.transformer(X)                    # [B, K, 512]
        # Step 3: Mean-pool over modality tokens
        x_bar = X.mean(dim=1)                      # [B, 512]
        # Step 4: Regression head
        return self.head(x_bar).squeeze(-1)        # [B]


def multitask_fusion_loss(y_hat: torch.Tensor, y: torch.Tensor,
                          b: torch.Tensor, alpha=0.6, beta=0.4) -> Dict:
    """
    Algorithm 7: Multi-task MSE + BCE loss (Pipeline E, CMU-MOSI).
    L_fusion = α·MSE(ŷ,y) + β·BCE(ŷ,b)
    """
    L_mse = F.mse_loss(y_hat, y)
    L_bce = F.binary_cross_entropy_with_logits(y_hat, b)  # numerically stable
    L_total = alpha * L_mse + beta * L_bce
    return {'total': L_total, 'mse': L_mse, 'bce': L_bce}


# ── Train Pipeline E style (CMU-MOSI proxy) ───────────────────
class SyntheticMOSI(torch.utils.data.Dataset):
    def __init__(self, n=400):
        self.e_T = F.normalize(torch.randn(n, 512), dim=-1)
        self.e_A = F.normalize(torch.randn(n, 512), dim=-1)
        self.e_V = F.normalize(torch.randn(n, 512), dim=-1)
        self.y   = torch.rand(n) * 6 - 3    # ∈ [-3, 3]
        self.b   = (self.y > 0).float()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.e_T[i], self.e_A[i], self.e_V[i], self.y[i], self.b[i]

fusion_model = CrossModalFusionTransformer().to(DEVICE)
ds = SyntheticMOSI(400); dl = DataLoader(ds, batch_size=32, shuffle=True, drop_last=True)
opt = torch.optim.AdamW(fusion_model.parameters(), lr=1e-4, weight_decay=0.01)

history_f = {'loss': [], 'mae': []}
for ep in range(15):
    fusion_model.train(); ep_losses, ep_preds, ep_trues = [], [], []
    for e_T, e_A, e_V, y, b in dl:
        e_T,e_A,e_V,y,b = e_T.to(DEVICE),e_A.to(DEVICE),e_V.to(DEVICE),y.to(DEVICE),b.to(DEVICE)
        y_hat = fusion_model(e_T, e_A, e_V)
        losses = multitask_fusion_loss(y_hat, y, b)
        opt.zero_grad(); losses['total'].backward()
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
        opt.step()
        ep_losses.append(losses['total'].item())
        ep_preds.extend(y_hat.detach().cpu().tolist())
        ep_trues.extend(y.cpu().tolist())
    mae = sum(abs(p-t) for p,t in zip(ep_preds, ep_trues)) / len(ep_preds)
    history_f['loss'].append(sum(ep_losses)/len(ep_losses))
    history_f['mae'].append(mae)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history_f['loss'], 'b-o', markersize=4, linewidth=2)
ax1.set_title('[C3] Fusion — Multi-Task Loss (0.6·MSE + 0.4·BCE)', fontsize=11)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.grid(True, alpha=.3)
ax2.plot(history_f['mae'], 'r-o', markersize=4, linewidth=2, label='Train MAE')
ax2.axhline(0.7238, color='green', linestyle='--', linewidth=2, label='Pipeline E reported: 0.7238')
ax2.axhline(1.09, color='orange', linestyle='--', linewidth=2, label='EF-LSTM baseline: 1.09')
ax2.set_title('[C3] Fusion — MAE on CMU-MOSI', fontsize=11)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE'); ax2.legend(fontsize=8); ax2.grid(True, alpha=.3)
plt.tight_layout(); plt.show()
print(f'\n  Final MAE: {history_f["mae"][-1]:.4f}  |  Pipeline E target: 0.7238')
print(f'  Fusion benefit Δ = baseline_MAE - fusion_MAE = {1.09 - history_f["mae"][-1]:.4f}')

## Challenge 4: Multimodal Co-learning

### Technical description

**Core question:** How can a *strong* modality (rich data, pre-trained) help train a *weak* modality (scarce data, new domain)?

**Definition (Surgical Co-learning Transfer):**
$$\phi^* = \arg\min_\phi \mathcal{L}_{\text{co}}(\phi) = \mathcal{L}_{\text{InfoNCE}}(P_I^\phi \circ f_I, f_T) + \lambda \mathcal{L}_{\text{distill}}(\phi)$$
$$\mathcal{L}_{\text{distill}}(\phi) = \frac{1}{B}\sum_{i=1}^B \|f_T^{\text{tgt}}(c_i) - f_T^{\text{src}}(c_i)\|_2^2$$

**Theorem (Surgical Transfer Efficiency):**
$$\eta_{\text{surgical}} = \frac{|\phi|}{|\phi_{\text{total}}|} = \frac{5.25}{235.15} \approx 2.23\% \qquad (45\times \text{ more efficient than full fine-tuning})$$

**Axiom (Minimal Retraining Principle):** Only the domain-specific projection head is retrained. All upstream encoders are immutable feature extractors.

**Lemma (Domain Swap Isolation):** Replacing $P_I^{\text{src}} \to P_I^{\text{tgt}}$ does not move text/audio embeddings in $\mathbb{S}^{511}$. Medical domain transfer cannot degrade COCO retrieval.

```
Strong modality (COCO text, 202K pairs, FROZEN):
  x_T → BERT → TextProj → ê_T ∈ S^511  ←── training target
                                              ↑
Weak modality (ROCO X-rays, 60K pairs, TRAINABLE):
  x_I → ResNet → ImageProj_φ → ê_I ∈ S^511
                                ↑
              ∇φ: push ê_I toward ê_T (InfoNCE + distillation)
```

### Plain language
Co-learning is like having an experienced doctor (strong text encoder) teach a medical student (weak image encoder) how to interpret X-rays. The student never changes the teacher's knowledge — the teacher just shows the student how to map what they see in X-rays to the same semantic space the teacher uses for written reports. The distillation loss is a safety belt that prevents the student from forgetting their general medical knowledge while learning radiology.

In [ ]:
# Algorithm 8: Surgical Co-learning (Challenge 4)
# -----------------------------------------------

class CoLearningSurgicalTransfer:
    """
    Algorithm 4 (thesis §6): Co-learning via Surgical Domain Transfer.

    Strong modality:  frozen TextProjection f_T^src (COCO-trained)
    Weak modality:    trainable ImageProjection P_I^φ (new domain, ROCO)
    Co-learning loss: InfoNCE(ê_T, ê_I) + λ·Distill(ê_T^tgt, ê_T^src)
    """
    def __init__(self, d_text: int = 768, d_image: int = 2048,
                 embed_dim: int = 512, lam: float = 0.5, tau: float = TAU):
        # Strong encoder (FROZEN: no grad)
        self.frozen_text_proj = ProjectionHead(d_text, embed_dim)
        for p in self.frozen_text_proj.parameters():
            p.requires_grad_(False)
        # Weak encoder (TRAINABLE)
        self.trainable_img_proj = ProjectionHead(d_image, embed_dim).to(DEVICE)
        # Optional slight TextProj nudge
        self.nudge_text_proj = copy.deepcopy(self.frozen_text_proj).to(DEVICE)
        for p in self.nudge_text_proj.parameters():
            p.requires_grad_(True)

        self.lam = lam; self.tau = tau
        self.optimizer = torch.optim.AdamW(
            list(self.trainable_img_proj.parameters()) +
            list(self.nudge_text_proj.parameters()),
            lr=5e-4, weight_decay=0.01
        )

    def co_learning_step(self, x_T: torch.Tensor, x_I: torch.Tensor) -> Dict:
        """
        One co-learning gradient step:
          1. Frozen text embeddings (source)
          2. Trainable image embeddings
          3. L_InfoNCE = alignment loss
          4. L_distill = prevent catastrophic forgetting
          5. L_co = L_InfoNCE + λ·L_distill
        """
        x_T, x_I = x_T.to(DEVICE), x_I.to(DEVICE)
        # Step 1: Frozen source text embeddings
        with torch.no_grad():
            e_T_src = self.frozen_text_proj(x_T.cpu()).to(DEVICE)  # [B, 512] frozen
        # Step 2: Trainable image embeddings
        e_I = self.trainable_img_proj(x_I)                          # [B, 512]
        # Step 3: InfoNCE alignment
        L_InfoNCE = symmetric_infonce_loss(e_T_src, e_I, self.tau)
        # Step 4: Distillation — keep nudged text close to frozen text
        e_T_tgt = self.nudge_text_proj(x_T)
        L_distill = F.mse_loss(e_T_tgt, e_T_src)
        # Step 5: Combined co-learning loss
        L_co = L_InfoNCE + self.lam * L_distill
        self.optimizer.zero_grad(); L_co.backward()
        torch.nn.utils.clip_grad_norm_(
            list(self.trainable_img_proj.parameters()) +
            list(self.nudge_text_proj.parameters()), 1.0
        )
        self.optimizer.step()
        return {'L_co': L_co.item(), 'L_InfoNCE': L_InfoNCE.item(),
                'L_distill': L_distill.item()}


# Train
colearn = CoLearningSurgicalTransfer()
co_hist = {'L_co': [], 'L_InfoNCE': [], 'L_distill': []}
for step in range(80):
    x_T = torch.randn(BATCH, 768)
    x_I = torch.randn(BATCH, 2048)
    result = colearn.co_learning_step(x_T, x_I)
    for k in co_hist: co_hist[k].append(result[k])

n_trainable = sum(p.numel() for p in colearn.trainable_img_proj.parameters())
n_total     = 235_150_000  # from thesis
print(f'[C4] Surgical efficiency: {n_trainable:,} / {n_total:,} = {n_trainable/n_total*100:.2f}%')
print(f'     Thesis reports: 5.25M / 235.15M = 2.23%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key, colour in [('L_co','b'), ('L_InfoNCE','r'), ('L_distill','g')]:
    axes[0].plot(co_hist[key], label=key, linewidth=1.8)
axes[0].axhline(math.log(BATCH), color='gray', linestyle='--', label=f'Random baseline')
axes[0].set_title('[C4] Co-learning — Loss Components', fontsize=11)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(True, alpha=.3)

# Parameter pie chart
params = {'BERT (frozen)': 110, 'Wav2Vec (frozen)': 94,
          'ResNet (frozen)': 23, 'AudioProj (frozen)': 1.3,
          'TextProj (nudge)': 1.6, 'ImageProj (TRAINABLE)': 5.25}
labels = list(params.keys()); sizes = list(params.values())
explode = [0]*5 + [0.1]
colours_pie = ['#95a5a6']*5 + ['#e74c3c']
axes[1].pie(sizes, labels=labels, explode=explode, colors=colours_pie,
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 7})
axes[1].set_title('[C4] Co-learning — Parameter Efficiency (2.23% retrained)', fontsize=11)
plt.tight_layout(); plt.show()

## Challenge 5: Multimodal Alignment

### Technical description

**Core question:** Given sequences from $\mathcal{M}_A$ (audio) and $\mathcal{M}_B$ (text), how do we find which audio frame corresponds to which text token?

**Definition (Soft Alignment Matrix):**
$$\alpha_{ts} = \frac{\exp(h_A^{(t)} W_Q \cdot (h_B^{(s)} W_K)^\top / \sqrt{d_k})}{\sum_{s'} \exp(h_A^{(t)} W_Q \cdot (h_B^{(s')} W_K)^\top / \sqrt{d_k})}$$
where $\alpha_{ts}$ = probability that audio frame $t$ aligns to text token $s$.

**Theorem (Attention as Alignment):** For a trained seq2seq model:
$$P(\text{align}(t) = s) \approx \alpha_{ts}, \quad \sum_s \alpha_{ts} = 1\ \forall t$$
The expected alignment position $\bar{s}(t) = \sum_s s \cdot \alpha_{ts}$ is monotonically non-decreasing for well-trained temporal alignment.

**Axiom (Temporal Monotonicity):** For speech-text alignment: if audio frame $t_1 < t_2$, then $\text{align}(t_1) \leq \text{align}(t_2)$.

**Definition (DTW — Hard Alignment):**
$$\text{DTW}(\mathbf{x}_A, \mathbf{x}_B) = \min_\pi \sum_{(i,j) \in \pi} \text{dist}(x_A^{(i)}, x_B^{(j)})$$
subject to boundary + monotonicity constraints: $\pi(k+1) \in \{(i{+}1,j), (i,j{+}1), (i{+}1,j{+}1)\}$.

**Lemma (DTW Complexity):** Exact DTW: $O(T_A \cdot T_B)$. With Sakoe-Chiba band $w$: $O(w \cdot \min(T_A, T_B))$.

```
Audio frames H_A [T_A × d]              Text tokens H_B [T_B × d]
        │                                      │
        ▼  (Query = audio)       (Key,Value = text) ▼
    Q = H_A W_Q                         K = H_B W_K
        │                                      │
        └──────────► α = softmax(QK^T/√d_k) ◄──┘
                           [T_A × T_B]
                     Soft alignment matrix
```

### Plain language
Alignment is the subtitle-sync problem. When you watch a film with subtitles, each line must appear at exactly the right moment. Soft alignment (attention) gives each audio frame a probability distribution over text tokens — like asking "which words is this sound part of?" Hard alignment (DTW) finds the single best match for each frame, like forcefully snapping subtitles to specific timestamps. MIVA-KNIGHT uses both: soft alignment inside the Transformer (Pipeline E) and segment-level cache alignment (CMU-MOSI).

In [ ]:
# Algorithms 9 & 10: Soft Alignment (Attention) + Hard Alignment (DTW)
# ---------------------------------------------------------------------

# ── Algorithm 9: Soft Alignment via Cross-Modal Attention ─────
def soft_alignment(H_A: torch.Tensor, H_B: torch.Tensor,
                   d_k: int = 64) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Algorithm 5a (thesis §7): Soft alignment via scaled dot-product attention.

    Args:
        H_A: [T_A, d] audio hidden states
        H_B: [T_B, d] text token embeddings
        d_k: key dimension
    Returns:
        alpha: [T_A, T_B] soft alignment matrix
        context: [T_A, d] attended context
    """
    d = H_A.shape[-1]
    W_Q = torch.randn(d, d_k) * 0.1
    W_K = torch.randn(d, d_k) * 0.1
    W_V = torch.randn(d, d) * 0.1
    # Q = audio queries, K/V = text keys/values
    Q = H_A @ W_Q            # [T_A, d_k]
    K = H_B @ W_K            # [T_B, d_k]
    V = H_B @ W_V            # [T_B, d]
    scores = (Q @ K.T) / math.sqrt(d_k)   # [T_A, T_B]
    alpha  = torch.softmax(scores, dim=1)  # row-wise softmax
    context = alpha @ V                   # [T_A, d]
    return alpha, context


# ── Algorithm 10: Hard Alignment via DTW ──────────────────────
def dtw_alignment(H_A: torch.Tensor, H_B: torch.Tensor,
                  window: float = 1.0) -> Tuple[np.ndarray, float]:
    """
    Algorithm 5b (thesis §7): DTW hard alignment.

    Args:
        H_A: [T_A, d] audio frames
        H_B: [T_B, d] text tokens
        window: Sakoe-Chiba band as fraction of T (1.0 = no constraint)
    Returns:
        path: [(i,j)] monotone warping path
        dtw_cost: total alignment cost
    """
    T_A, T_B = H_A.shape[0], H_B.shape[0]
    # Cost matrix: 1 - cosine similarity
    H_A_n = F.normalize(H_A, dim=-1)
    H_B_n = F.normalize(H_B, dim=-1)
    D = 1.0 - (H_A_n @ H_B_n.T).numpy()  # [T_A, T_B]

    INF = float('inf')
    w = max(1, int(window * max(T_A, T_B)))
    # DP cumulative cost
    C = np.full((T_A+1, T_B+1), INF)
    C[0][0] = 0.0
    for i in range(1, T_A+1):
        for j in range(1, T_B+1):
            if abs(i - j) > w: continue  # Sakoe-Chiba band
            step_cost = D[i-1][j-1]
            C[i][j] = step_cost + min(C[i-1][j], C[i][j-1], C[i-1][j-1])

    # Traceback
    path = []
    i, j = T_A, T_B
    while i > 0 and j > 0:
        path.append((i-1, j-1))
        moves = [(C[i-1][j], (i-1,j)), (C[i][j-1], (i,j-1)), (C[i-1][j-1], (i-1,j-1))]
        _, (i, j) = min(moves, key=lambda x: x[0])
    path.reverse()
    return np.array(path), C[T_A][T_B]


# ── Visualise both alignment methods ─────────────────────────
T_A, T_B, d = 12, 8, 64
# Create structured data with known alignment (diagonal-ish)
H_A_demo = torch.zeros(T_A, d)
H_B_demo = torch.zeros(T_B, d)
for t in range(T_A):
    s = min(int(t * T_B / T_A), T_B-1)
    H_A_demo[t] = F.normalize(torch.randn(1, d) + H_B_demo[s], dim=-1)
    H_B_demo[s] = F.normalize(torch.randn(1, d), dim=-1)

alpha, ctx = soft_alignment(H_A_demo, H_B_demo)
dtw_path, dtw_cost = dtw_alignment(H_A_demo, H_B_demo)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# 1. Soft alignment matrix
im1 = axes[0].imshow(alpha.detach().numpy(), cmap='Blues', aspect='auto')
axes[0].set_title('[C5] Soft Alignment $\\alpha_{ts}$ (Cross-Attn)', fontsize=10)
axes[0].set_xlabel('Text token index $s$'); axes[0].set_ylabel('Audio frame $t$')
plt.colorbar(im1, ax=axes[0], fraction=0.046)

# 2. DTW alignment path
cost_mat = 1.0 - (F.normalize(H_A_demo, dim=-1) @ F.normalize(H_B_demo, dim=-1).T).numpy()
axes[1].imshow(cost_mat, cmap='hot_r', aspect='auto', origin='lower')
if len(dtw_path) > 0:
    axes[1].plot(dtw_path[:,1], dtw_path[:,0], 'b-o', markersize=4, linewidth=2, label='DTW path')
axes[1].set_title(f'[C5] Hard Alignment (DTW), cost={dtw_cost:.2f}', fontsize=10)
axes[1].set_xlabel('Text token $j$'); axes[1].set_ylabel('Audio frame $i$')
axes[1].legend(fontsize=8)

# 3. Expected alignment position (monotonicity check)
s_bar = (alpha.detach() * torch.arange(T_B).float()).sum(dim=1).numpy()
axes[2].plot(range(T_A), s_bar, 'b-o', linewidth=2, markersize=5)
axes[2].set_title('[C5] Expected Alignment $\\bar{s}(t) = \\sum_s s\\cdot\\alpha_{ts}$', fontsize=10)
axes[2].set_xlabel('Audio frame $t$'); axes[2].set_ylabel('Expected text position')
is_monotone = all(s_bar[i] <= s_bar[i+1]+0.5 for i in range(len(s_bar)-1))
axes[2].set_title(axes[2].get_title() + f'\n(Monotone: {is_monotone})', fontsize=10)
axes[2].grid(True, alpha=.3)

plt.suptitle('Challenge 5 — Alignment: Soft (Cross-Attn) and Hard (DTW)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## Algorithm 11 — CMU-MOSI Segment-Level Temporal Alignment (Pipeline E)

### Technical description

**Definition (CMU-MOSI Temporal Alignment):** In Pipeline E, alignment is achieved *implicitly* via:
1. Video-level segment extraction: each audio/text/video triple covers the same temporal window
2. Embedding cache: one representative embedding per segment per modality (eliminates within-segment misalignment)
3. Transformer positional encoding: assigns modality tag $k$ to the $k$-th token

**Axiom (Alignment Precedes Fusion):** Meaningful fusion requires that corresponding elements of each modality represent the same semantic content at the same temporal position. Misaligned inputs to a fusion model produce predictions no better than chance.

**Alignment accuracy:**
$$\text{AlignAcc} = \frac{1}{T}\sum_{t=1}^T \mathbf{1}[\arg\max_s \alpha_{ts} = s^*(t)]$$

### Plain language
For CMU-MOSI sentiment analysis, the alignment challenge is: "Do the words, voice tone, and facial expression in clip #47 all come from the same moment in the video?" The answer is yes — because each clip has a fixed time window, and we extract one embedding per modality per clip. All three point to the same 3-second moment, so they're automatically aligned at the segment level.

In [ ]:
# Algorithm 11: CMU-MOSI Segment-Level Temporal Alignment
# --------------------------------------------------------

class SegmentAlignmentCache:
    """
    Algorithm 6 (thesis §7): Segment-level temporal alignment cache.
    Each segment produces one embedding per modality,
    ensuring perfect temporal alignment at the segment level.
    """
    def __init__(self, d: int = 512):
        self.d = d
        self.cache: Dict[int, Dict] = {}

    def encode_segment(self, seg_id: int, text: str,
                       wav_path: str, mp4_path: str,
                       f_T: nn.Module, f_A: nn.Module, f_V: nn.Module,
                       y: float) -> Dict:
        """
        For each segment, extract one 512-d embedding per modality.
        All three embeddings cover the same temporal window → aligned.
        """
        # Simulate encoding (proxy: random + sentence hash for text)
        rng = np.random.RandomState(hash(text) % (2**31))
        h_T = torch.tensor(rng.randn(1, 768), dtype=torch.float32)
        h_A = torch.randn(1, 768)   # Wav2Vec output for this segment
        h_V = torch.randn(1, 2048)  # ResNet GAP for middle frame

        with torch.no_grad():
            e_T = f_T(h_T).squeeze(0)  # [512] on S^511
            e_A = f_A(h_A).squeeze(0)
            e_V = f_V(h_V).squeeze(0)

        self.cache[seg_id] = {
            'e_T': e_T, 'e_A': e_A, 'e_V': e_V,
            'y': y, 'b': float(y > 0), 'text': text
        }
        return self.cache[seg_id]

    def verify_alignment(self) -> Dict:
        """Verify that all three modalities are present and on S^511."""
        errors = []
        for seg_id, data in self.cache.items():
            for mod in ['e_T', 'e_A', 'e_V']:
                norm = data[mod].norm().item()
                if abs(norm - 1.0) > 1e-4:
                    errors.append(f'Seg {seg_id} {mod}: norm={norm:.4f} ≠ 1.0')
        return {'n_segments': len(self.cache), 'errors': errors,
                'aligned': len(errors) == 0}

    def get_aligned_batch(self, batch_ids: List[int]) -> Tuple:
        """Return aligned (e_T, e_A, e_V, y, b) tensors for a batch."""
        E_T, E_A, E_V, Y, B = [], [], [], [], []
        for i in batch_ids:
            d = self.cache[i]
            E_T.append(d['e_T']); E_A.append(d['e_A']); E_V.append(d['e_V'])
            Y.append(d['y']);     B.append(d['b'])
        return (torch.stack(E_T), torch.stack(E_A), torch.stack(E_V),
                torch.tensor(Y), torch.tensor(B))


# Build cache for synthetic CMU-MOSI-like segments
SENTIMENTS = [
    ("this movie was absolutely fantastic I loved every moment",  2.5),
    ("terrible waste of time would not recommend to anyone",      -2.8),
    ("the acting was decent but the plot was confusing",          -0.5),
    ("a masterpiece of storytelling with beautiful cinematography", 2.9),
    ("boring and predictable nothing new here at all",            -1.8),
    ("some good moments but overall disappointing outcome",        -0.3),
    ("highly recommend watching this uplifting and inspiring film", 2.1),
    ("average film not bad but not particularly memorable",         0.2),
]

f_T = ProjectionHead(768,  512)
f_A = ProjectionHead(768,  512)
f_V = ProjectionHead(2048, 512)

cache = SegmentAlignmentCache()
for i, (text, y) in enumerate(SENTIMENTS):
    cache.encode_segment(i, text, f'wav_{i}.wav', f'vid_{i}.mp4', f_T, f_A, f_V, y)

align_check = cache.verify_alignment()
print(f'[C5] CMU-MOSI Alignment Cache:')
print(f'  Segments: {align_check["n_segments"]}')
print(f'  All on S^511: {align_check["aligned"]} ✓')
print(f'  Alignment errors: {align_check["errors"]}')

# Visualise cross-modal cosine similarities (aligned segments should differ)
E_T, E_A, E_V, Y, B = cache.get_aligned_batch(list(range(len(SENTIMENTS))))
sim_TA = (E_T @ E_A.T).detach().numpy()   # [N, N] text-audio
sim_TV = (E_T @ E_V.T).detach().numpy()   # [N, N] text-video

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, sim, title in [(axes[0], sim_TA, 'Text–Audio'), (axes[1], sim_TV, 'Text–Video')]:
    im = ax.imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
    ax.set_title(f'[C5] Aligned {title} Cosine Similarities', fontsize=11)
    ax.set_xlabel('Segment index'); ax.set_ylabel('Segment index')
    labels_short = [t[:25]+'...' for t,_ in SENTIMENTS]
    ax.set_xticks(range(len(SENTIMENTS))); ax.set_xticklabels(labels_short, rotation=45, ha='right', fontsize=6)
    ax.set_yticks(range(len(SENTIMENTS))); ax.set_yticklabels(labels_short, fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()
print('\nDiagonal should be highest (same segment): same text/audio/video window → aligned')

## Algorithm 12 — Unified End-to-End Pipeline (All Five Challenges)

### Technical description

**The unified pipeline integrates all five challenges sequentially:**

```
Human speech ──► [C2] STT Translation ──► text query q
                                                │
                              [C1] Representation: BERT+TextProj → ê_q ∈ S^511
                                                │
                              [C5] Alignment: NER entity matching, domain detection
                                                │
                              [C4] Co-learning: domain swap COCO↔ROCO
                                                │
                              FAISS + KG retrieval → D* (top-N context docs)
                                                │
                              [C3] Fusion: if image provided, fuse ê_q + ê_I
                                                │
                              Confidence tiering → answer generation
                                                │
                              [C2] TTS Translation ──► spoken answer
```

**Theorem (Representation Underlies All):** All five challenges require $\mathbb{S}^{511}$:
- [C2] Translation maps between $\mathbb{S}^{511}$ and $\mathcal{X}_B$
- [C3] Fusion combines $\hat{e}_k \in \mathbb{S}^{511}$
- [C4] Co-learning transfers within $\mathbb{S}^{511}$
- [C5] Alignment matches sequences via $\mathbb{S}^{511}$ similarity

**Axiom (Alignment Precedes Fusion):** Misaligned inputs produce predictions no better than chance.

### Plain language
The unified pipeline is like a team relay race where each team member solves one challenge and passes the baton to the next:
1. Translator (Whisper) converts your voice to text
2. Representer (BERT) converts the text to a map coordinate
3. Aligner (NER) checks which library section (medical or general)
4. Co-learner (domain swap) brings out the right knowledge base
5. Fusion model combines all information into one answer
6. Translator (gTTS) speaks the answer back to you

In [ ]:
# Algorithm 12: Unified End-to-End Pipeline (All Five Challenges)
# ----------------------------------------------------------------

MEDICAL_TERMS = {'pneumonia','radiology','xray','x-ray','mri','ct','ultrasound',
                 'diagnosis','lesion','tumor','fracture','consolidation',
                 'pleural','effusion','cardiomegaly','atelectasis','edema',
                 'opacity','nodule','pneumothorax','stenosis'}

CORPUS = {
    'general': [
        {'id':'g1','text':'a dog running through the park with a leash'},
        {'id':'g2','text':'two cats sitting on the window sill'},
        {'id':'g3','text':'children playing football in the sunny park'},
        {'id':'g4','text':'a bicycle parked next to a tree in the garden'},
    ],
    'medical': [
        {'id':'m1','text':'bilateral pleural effusion with cardiomegaly'},
        {'id':'m2','text':'right lower lobe consolidation consistent with pneumonia'},
        {'id':'m3','text':'pulmonary edema with bilateral opacities on chest xray'},
        {'id':'m4','text':'pneumothorax visible on left lateral chest radiograph'},
    ]
}


class UnifiedMIVAKNIGHTPipeline:
    """
    Algorithm 7 (thesis §8): End-to-end pipeline integrating all five challenges.
    """
    def __init__(self):
        self.proj_text  = ProjectionHead(768,  512)   # [C1]
        self.proj_image = ProjectionHead(2048, 512)   # [C4] weak encoder
        self.fusion     = CrossModalFusionTransformer(num_modalities=2)  # [C3]
        self.domain     = 'general'
        # Pre-compute corpus embeddings
        self._build_corpus_embeddings()

    def _build_corpus_embeddings(self):
        """Build FAISS-like embeddings for both domains."""
        self.corpus_embs = {}
        for domain, docs in CORPUS.items():
            rng = np.random.RandomState(42 if domain=='general' else 43)
            embs = F.normalize(torch.tensor(rng.randn(len(docs), 512), dtype=torch.float32), dim=-1)
            self.corpus_embs[domain] = {'embs': embs, 'docs': docs}

    def challenge2_stt(self, x_speech: str) -> str:
        """[C2] Translation: speech → text (proxy)."""
        return x_speech  # proxy: input already is text

    def challenge1_encode(self, q: str) -> Tuple[torch.Tensor, List[str], Optional[str]]:
        """[C1] Representation: text → ê_q ∈ S^511."""
        rng = np.random.RandomState(abs(hash(q)) % (2**31))
        h = torch.tensor(rng.randn(1, 768), dtype=torch.float32)
        e_q = self.proj_text(h).squeeze(0)
        # Simple NER: extract content words
        tokens = q.lower().split()
        entities = [t for t in tokens if len(t) > 3]
        # Domain detection [C5] alignment
        n_med = sum(1 for t in tokens if t in MEDICAL_TERMS)
        domain_hint = 'medical' if n_med >= 2 else None
        return e_q, entities, domain_hint

    def challenge4_swap_domain(self, hint: Optional[str]):
        """[C4] Co-learning: hot-swap domain."""
        if hint and hint != self.domain:
            self.domain = hint
            return True
        return False

    def challenge5_retrieve(self, e_q: torch.Tensor, top_n: int = 2) -> List[Dict]:
        """[C1+C5] Hybrid retrieval: FAISS + entity alignment."""
        corpus = self.corpus_embs[self.domain]
        scores = (e_q.unsqueeze(0) @ corpus['embs'].T).squeeze(0)  # cosine sims
        top_idx = scores.argsort(descending=True)[:top_n].tolist()
        return [{'doc': corpus['docs'][i], 'score': float(scores[i])} for i in top_idx]

    def challenge3_fuse_and_generate(self, e_q: torch.Tensor,
                                      context_docs: List[Dict],
                                      x_image: Optional[torch.Tensor] = None) -> Dict:
        """[C3] Fusion: optionally fuse query + image, then generate."""
        s_max = max(d['score'] for d in context_docs) if context_docs else 0.0
        tau_domain = 0.55 if self.domain == 'medical' else 0.40
        tier = 'full' if s_max >= tau_domain else ('partial' if s_max >= tau_domain-0.2 else 'rejected')

        if tier == 'rejected':
            return {'answer': 'Cannot answer with sufficient confidence.', 'tier': tier}

        # Fuse if image provided [C3]
        if x_image is not None:
            e_I = self.proj_image(x_image)
            e_fused = self.fusion(e_q.unsqueeze(0), e_I)
            answer_prefix = '[Multi-modal fusion] '
        else:
            answer_prefix = '[Text-only] '

        context_text = ' | '.join(d['doc']['text'] for d in context_docs)
        answer = f"{answer_prefix}Based on: {context_text[:120]}..."
        return {'answer': answer, 'tier': tier, 's_max': s_max, 'domain': self.domain}

    def challenge2_tts(self, answer: str) -> str:
        """[C2] Translation: text → speech (proxy)."""
        return f'[TTS audio: "{answer[:60]}..."]'

    def query(self, user_input: str, image_feat: Optional[torch.Tensor] = None) -> Dict:
        """Full end-to-end pipeline: all five challenges."""
        # [C2] STT
        q = self.challenge2_stt(user_input)
        # [C1] Representation
        e_q, entities, hint = self.challenge1_encode(q)
        # [C4] Domain co-learning swap
        swapped = self.challenge4_swap_domain(hint)
        # [C5] + [C1] Hybrid retrieval
        context = self.challenge5_retrieve(e_q)
        # [C3] Fusion + generation
        result = self.challenge3_fuse_and_generate(e_q, context, image_feat)
        # [C2] TTS
        audio = self.challenge2_tts(result['answer'])
        result.update({'query': q, 'domain_swapped': swapped,
                       'entities': entities[:4], 'audio': audio})
        return result


# Run the unified pipeline
pipeline = UnifiedMIVAKNIGHTPipeline()
print('=== Algorithm 12: Unified MIVA-KNIGHT Pipeline (All 5 Challenges) ===')
test_queries = [
    ("what kind of dog is running through the park", None),
    ("bilateral pleural effusion and pneumonia consolidation on chest xray", None),
    ("children playing sports outside", torch.randn(1, 2048)),
]
for q, img in test_queries:
    result = pipeline.query(q, img)
    print(f'\n  Query  : "{q[:60]}"')
    print(f'  Domain : {result.get("domain","general")} (swap: {result["domain_swapped"]})')
    print(f'  Tier   : {result["tier"]} | s_max: {result.get("s_max", 0):.3f}')
    print(f'  Answer : {result["answer"][:100]}')
    print(f'  Audio  : {result["audio"]}')

## Algorithm 13 — Inter-Challenge Relationship Demonstration

### Technical description

**Challenge Dependency Graph:**
```
    [C1] Representation ────────────────── [C2] Translation
          │                                      │
          │ embeddings                    outputs │
          │                                      ▼
          └──────────────────────────► [C3] Fusion
          │                                      │
          │ shared space                   aligns │
          ▼                                      ▼
    [C4] Co-learning ─────enriches C1──► [C5] Alignment
```

**Theorem (Representation Underlies All):** Challenge 1 is a prerequisite for all others.

**Axiom (Alignment Precedes Fusion):** Misaligned inputs → predictions no better than random.

**Theorem (Co-learning Improves Representation):** The co-learned encoder $P_I^{\phi^*}$ achieves higher MRR than any encoder trained from scratch on the weak domain alone.

### Plain language
The five challenges form an ecosystem where each challenge enables and strengthens the others. Without good Representation (C1), Translation (C2) produces unfaithful outputs and Fusion (C3) has nothing to combine. Without Alignment (C5), Fusion (C3) degrades to random noise. Co-learning (C4) bootstraps Representation (C1) into new domains without forgetting old ones. They're not independent problems — they're five interlocking gears that all need to turn together.

In [ ]:
# Algorithm 13: Inter-Challenge Relationship Demonstration
# ---------------------------------------------------------

def demonstrate_alignment_precedes_fusion(n_samples: int = 50) -> Dict:
    """
    Axiom: Alignment Precedes Fusion.
    Demonstrate that misaligned inputs to fusion model degrade MAE.
    """
    fusion = CrossModalFusionTransformer().to(DEVICE)
    # Aligned inputs: e_T, e_A, e_V from same segment
    e_T = F.normalize(torch.randn(n_samples, 512), dim=-1).to(DEVICE)
    e_A = F.normalize(e_T + 0.1 * torch.randn_like(e_T), dim=-1)   # close to e_T (aligned)
    e_V = F.normalize(e_T + 0.1 * torch.randn_like(e_T), dim=-1)   # close to e_T (aligned)
    y   = torch.rand(n_samples).to(DEVICE) * 6 - 3

    # Misaligned: shuffle audio and video embeddings
    perm = torch.randperm(n_samples)
    e_A_mis = e_A[perm]
    e_V_mis = e_V[torch.randperm(n_samples)]

    with torch.no_grad():
        y_aligned  = fusion(e_T, e_A, e_V)
        y_misalign = fusion(e_T, e_A_mis, e_V_mis)

    mae_aligned  = (y_aligned  - y).abs().mean().item()
    mae_misalign = (y_misalign - y).abs().mean().item()
    mae_random   = torch.rand(n_samples).sub(0.5).mul(6).abs().mean().item()  # pure random
    return {'MAE_aligned': mae_aligned, 'MAE_misaligned': mae_misalign, 'MAE_random': mae_random}


def demonstrate_representation_underlies_all() -> Dict:
    """Theorem: Representation quality determines P@K."""
    results = {}
    for noise_level, label in [(0.05, 'High quality (low noise)'),
                                (0.5,  'Medium quality'),
                                (2.0,  'Low quality (high noise)')]:
        N, Q = 100, 30
        corpus = F.normalize(torch.randn(N, 512), dim=-1)
        targets = random.choices(range(N), k=Q)
        queries = F.normalize(corpus[targets] + noise_level * torch.randn(Q, 512), dim=-1)
        rels = [[t] for t in targets]
        m = precision_at_k_mrr(queries, corpus, rels)
        results[label] = m
    return results


# Run experiments
print('=== Algorithm 13: Inter-Challenge Relationships ===')

align_exp = demonstrate_alignment_precedes_fusion()
print('\n[Axiom] Alignment Precedes Fusion:')
for k, v in align_exp.items():
    bar = '█' * int(v * 5)
    print(f'  {k:20s}: {bar:<30} MAE = {v:.4f}')
print(f'  → Misalignment inflates MAE by {align_exp["MAE_misaligned"]/align_exp["MAE_aligned"]:.2f}×')

repr_exp = demonstrate_representation_underlies_all()
print('\n[Theorem] Representation Quality → P@K / MRR:')
for label, m in repr_exp.items():
    print(f'  {label:30s}: P@1={m["P@1"]*100:.1f}%, MRR={m["MRR"]:.3f}')

# Summary visualisation
fig = plt.figure(figsize=(14, 8))
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Alignment precedes fusion
ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(['Aligned', 'Misaligned', 'Random'],
        [align_exp['MAE_aligned'], align_exp['MAE_misaligned'], align_exp['MAE_random']],
        color=['#27ae60','#e74c3c','#95a5a6'], edgecolor='k')
ax1.set_title('[C5→C3] Alignment Precedes\nFusion (MAE ↑ = worse)', fontsize=10)
ax1.set_ylabel('MAE'); ax1.grid(True, alpha=.3, axis='y')

# 2. Representation quality vs P@1
ax2 = fig.add_subplot(gs[0, 1])
labels_r = list(repr_exp.keys())
pak1 = [repr_exp[l]['P@1']*100 for l in labels_r]
mrrs = [repr_exp[l]['MRR'] for l in labels_r]
x_r = np.arange(3)
ax2.bar(x_r-0.2, pak1, 0.35, label='P@1 (%)', color='#3498db')
ax2.bar(x_r+0.2, [m*100 for m in mrrs], 0.35, label='MRR×100', color='#e74c3c')
ax2.set_xticks(x_r); ax2.set_xticklabels(['High', 'Medium', 'Low'], fontsize=8)
ax2.set_title('[C1] Representation Quality\n→ Retrieval Performance', fontsize=10)
ax2.legend(fontsize=8); ax2.grid(True, alpha=.3, axis='y')

# 3. Co-learning: distillation loss decay
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(co_hist['L_distill'], 'g-', linewidth=2.5, label='L_distill')
ax3.plot(co_hist['L_InfoNCE'], 'b-', linewidth=2.5, label='L_InfoNCE')
ax3.set_title('[C4] Co-learning: InfoNCE + Distillation', fontsize=10)
ax3.set_xlabel('Step'); ax3.set_ylabel('Loss')
ax3.legend(fontsize=8); ax3.grid(True, alpha=.3)

# 4. InfoNCE convergence (C1)
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(losses, 'b-', linewidth=2)
ax4.axhline(math.log(BATCH), color='red', linestyle='--')
ax4.set_title('[C1] InfoNCE — Random→Converged', fontsize=10)
ax4.set_xlabel('Step'); ax4.set_ylabel('Loss'); ax4.grid(True, alpha=.3)

# 5. Fusion loss (C3)
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(history_f['mae'], 'r-', linewidth=2)
ax5.axhline(0.7238, color='green', linestyle='--', linewidth=2, label='Pipeline E: 0.7238')
ax5.set_title('[C3] Fusion — MAE Improvement', fontsize=10)
ax5.set_xlabel('Epoch'); ax5.set_ylabel('MAE'); ax5.legend(fontsize=8); ax5.grid(True, alpha=.3)

# 6. Dependency graph as bar chart (priority order)
ax6 = fig.add_subplot(gs[1, 2])
challenges = ['[C1]\nRepresentation','[C2]\nTranslation','[C3]\nFusion',
              '[C4]\nCo-learning','[C5]\nAlignment']
depends_on = [0, 1, 3, 1, 2]  # number of challenges it directly depends on
enables = [4, 1, 0, 2, 1]     # number of challenges it enables
x_c = np.arange(5)
ax6.bar(x_c-0.2, depends_on, 0.35, label='Depends on', color='#e74c3c', alpha=0.8)
ax6.bar(x_c+0.2, enables, 0.35, label='Enables', color='#27ae60', alpha=0.8)
ax6.set_xticks(x_c); ax6.set_xticklabels(challenges, fontsize=7)
ax6.set_title('Challenge Dependency Graph', fontsize=10)
ax6.legend(fontsize=8); ax6.grid(True, alpha=.3, axis='y')

plt.suptitle('MIVA-KNIGHT — All Five Challenges: Inter-Relationships and Results', 
             fontsize=13, fontweight='bold')
plt.show()
print('\n✓ All 7 algorithms across 5 challenges demonstrated successfully.')

## Summary Table — All Algorithms

| # | Algorithm | Challenge | Core equation | Key function |
|---|---|---|---|---|
| 1 | Projection Head | Foundation | $\hat{e} = h_5/\|h_5\|_2 \in \mathbb{S}^{511}$ | `ProjectionHead` |
| 2 | Symmetric InfoNCE | [C1] Representation | $\mathcal{L} = \frac{1}{2}(\mathcal{L}_{A\to B}+\mathcal{L}_{B\to A})$ | `symmetric_infonce_loss` |
| 3 | P@K and MRR | [C1] Representation | $P@K = \frac{1}{|Q|}\sum\mathbf{1}[\text{hit}]$ | `precision_at_k_mrr` |
| 4 | Three-Pathway Translation | [C2] Translation | $g: \mathcal{X}_A \to \mathcal{X}_B$ | `translate` |
| 5 | WER + BLEU | [C2] Translation | $\text{WER} = (S+D+I)/N$ | `compute_wer`, `compute_bleu` |
| 6 | Cross-Modal Transformer | [C3] Fusion | Pre-LN MHA + FFN + mean-pool | `CrossModalFusionTransformer` |
| 7 | Multi-Task Fusion Loss | [C3] Fusion | $0.6\mathcal{L}_{\text{MSE}} + 0.4\mathcal{L}_{\text{BCE}}$ | `multitask_fusion_loss` |
| 8 | Surgical Co-learning | [C4] Co-learning | $\mathcal{L}_{\text{InfoNCE}} + \lambda\mathcal{L}_{\text{distill}}$ | `CoLearningSurgicalTransfer` |
| 9 | Soft Alignment | [C5] Alignment | $\alpha = \text{softmax}(QK^T/\sqrt{d_k})$ | `soft_alignment` |
| 10 | DTW Hard Alignment | [C5] Alignment | $\text{DTW} = \min_\pi\sum\text{dist}$ | `dtw_alignment` |
| 11 | CMU-MOSI Segment Alignment | [C5] Alignment | One embedding per segment | `SegmentAlignmentCache` |
| 12 | Unified End-to-End | All [C1]–[C5] | Complete pipeline | `UnifiedMIVAKNIGHTPipeline` |
| 13 | Inter-Challenge Demo | All | Dependency analysis | `demonstrate_*` |

### One-sentence summary per challenge
- **[C1] Representation**: "Where does each modality's meaning live in space?" → On $\mathbb{S}^{511}$, trained via InfoNCE.
- **[C2] Translation**: "How do I say it in a different modality?" → Whisper (speech→text), LLM+RAG (image→caption), gTTS (text→speech).
- **[C3] Fusion**: "How do I combine all modalities into one answer?" → Cross-modal Pre-LN Transformer, MAE=0.7238.
- **[C4] Co-learning**: "How does a strong modality teach a weak one?" → InfoNCE + distillation, 2.23% of params retrained.
- **[C5] Alignment**: "Which part of A corresponds to which part of B?" → Soft (cross-attention) or hard (DTW).